# Week 5 Lab 3: Logistic Regression (Titanic Survival & Pipelines)

**Goal**: Predict whether a passenger **Survived (1)** or **Perished (0)** based on their social status and demographics using the Titanic dataset.

> **Why this lab matters**:
> Logistic Regression introduces **Binary Classification**. It teaches models to output probabilities (0 to 100%) rather than infinite continuous numbers. This is the foundation for all decision-making AI.

> **Structure**:
> We follow the **6-Phase Professional Workflow** and use a `ColumnTransformer` to handle both numerical data and categorical text data simultaneously. We will test the model's reliability using **Cross-Validation**.

---
## Foreword
In this lab, we use the legendary **Kaggle Titanic Dataset**.

1. **Phase 1: Splitting**
2. **Phase 2: Preprocessing (ColumnTransformers)**
3. **Phase 3: Assembly (Pipeline)**
4. **Phase 4: Training**
5. **Phase 5: Evaluation (Accuracy, Probabilities & Cross-Validation)**
6. **Phase 6: Optimization**

### 1.1 Import Dependencies & Load Data
**Concept**: We fetch the Titanic data and focus on key features: Passenger Class (`Pclass`), Gender (`Sex`), and `Age`.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Load Titanic Dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Preprocess: Keep key features and drop missing values for simplicity
df = df[['Survived', 'Pclass', 'Sex', 'Age']].dropna()

X = df[['Pclass', 'Sex', 'Age']]
y = df['Survived']

---
### 1.2 Phase 1: Data Splitting
**Concept**: We secure 20% of the passenger data for our unbiased survival final exam.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
#### Theory: Logistic Regression & Dummy Variables
When One-Hot Encoding binary features like `Sex` ('male', 'female'), creating two columns is redundant (if not male, must be female). We drop the first column to prevent the "Dummy Variable Trap."

| Component | Target | Function |
| :--- | :--- | :--- |
| `StandardScaler()` | `['Pclass', 'Age']` | Logistic Regression converges faster when numbers are scaled. |
| `OneHotEncoder(drop='first')` | `['Sex']` | Converts 'male'/'female' into a single binary 1/0 column. |

### 1.3 Phase 2 & 3: Preprocessing & Assembly
**Concept**: We map our operations to the exact columns they belong to.
**Solution**: We use `ColumnTransformer` for routing and wrap it in our `Pipeline`.


In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['Pclass', 'Age']),
    ('cat', OneHotEncoder(drop='first'), ['Sex'])
])

workflow = Pipeline([
    ('pre', preprocessor),
    ('model', LogisticRegression())
])

---
### 1.4 Phase 4: Training
**Concept**: We train the model to find the optimal 'S-Curve' (Sigmoid) that separates survivors from those who perished.
**Solution**: Call `.fit()` on the entire pipeline.


In [ ]:
workflow.fit(X_train, y_train)

---
### 1.5 Phase 5: Evaluation (Accuracy & Probabilities)
**Concept**: We check the model's accuracy on the unseen test data. More importantly, we look at the raw probabilities the Sigmoid gives us.
**Solution**: We use `.predict()` for the final 0/1 decision, and `.predict_proba()` to see the percentage certainty.


In [ ]:
y_pred = workflow.predict(X_test)
y_prob = workflow.predict_proba(X_test)

print(f"Survival Prediction Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"\nSample Prediction Probability for First Test Passenger:")
print(f"Perish: {y_prob[0, 0]:.2%} | Survive: {y_prob[0, 1]:.2%}")

---
### 1.6 Phase 5: Cross-Validation
**Concept**: Does the model perform consistently across all variations of the Titanic passenger lists?
**Solution**: We run a 5-fold cross-validation on the pipeline, using standard `accuracy` as the metric.


In [ ]:
scores = cross_val_score(workflow, X, y, scoring='accuracy', cv=5)
print("Accuracy Scores across 5 folds:", np.round(scores, 3))
print(f"\nAverage CV Accuracy: {scores.mean():.2%}")
print(f"Standard Deviation: {scores.std():.2%} (How much the accuracy fluctuates)")

> **Observation**: The cross-validation score reveals if the model's high accuracy was just luck. A stable accuracy around 78-80% is typical for this subset of features.

**Task 1**: In the `ColumnTransformer` in Phase 2, change `drop='first'` to `None` in the `OneHotEncoder`. Rerun the cross-validation. Does creating two dummy variables instead of one crash the model or change the accuracy significantly? Why or why not?

<details>
<summary><strong> Click here for Solution (Try it yourself first!)</strong></summary>

If you remove `drop='first'`, `OneHotEncoder` creates two columns: `Sex_female` and `Sex_male`. The accuracy will likely remain identical.

Modern Scikit-Learn's `LogisticRegression` includes $L_2$ Regularization by default (the `C=1.0` parameter), which gracefully handles the collinearity (Dummy Variable Trap). However, for mathematical purity and slightly faster training, dropping the first column is best practice.
</details>

---
### Summary
You've just built an AI that can predict historical outcomes with high consistency across cross-validation folds. Notice how the model doesn't just guess Yes/No; it calculates a **Probability** before making the final call!